# Lekce 05 - Agentní RAG


## Nastavení

Tento notebook demonstruje vzor Agentic RAG (Retrieval-Augmented Generation) pomocí Microsoft Agent Frameworku.

**Požadavky:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — váš koncový bod služby Azure AI Search
- `AZURE_SEARCH_API_KEY` — váš API klíč pro Azure AI Search
- Nasazení Azure OpenAI nakonfigurované prostřednictvím proměnných prostředí
- Autentizace v Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Co je Agentic RAG?

Tradiční RAG následuje pevně daný postup: vyhledat dokumenty a poté vygenerovat odpověď. **Agentic RAG** jde dál tím, že dává agentovi autonomii rozhodnout, **kdy** a **jak** vyhledat informace.

S Agentic RAG může agent:
- **Rozhodnout**, zda je před odpovědí potřeba vyhledávání
- **Vybrat**, který zdroj dat nebo nástroj použít pro dotaz
- **Zhodnotit** získané výsledky a provést další vyhledávání, pokud první pokus není dostatečný
- **Kombinovat** informace z několika kroků vyhledávání do koherentní odpovědi

To činí agenta flexibilnějšího a přesnějšího ve srovnání s pevně daným postupem „vyhledat a poté vygenerovat“.


## Vytvoření nástroje pro vyhledávání

V Agentic RAG jsou externí zdroje dat zabaleny jako **nástroje**, které může agent volat podle potřeby. To umožňuje agentovi považovat získávání informací za další akci, kterou může provést, místo povinného kroku.

Níže definujeme znalostní databázi o cestování a zpřístupníme ji jako nástroj, který může agent volat k vyhledání informací o cílových místech.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Vytváření RAG agenta

Nyní vytvoříme agenta, který je instruován **vždy nejprve získat informace před odpovědí**. Agent používá nástroj `search_travel_knowledge`, aby podložil své odpovědi v databázi znalostí, místo aby spoléhal na svá vlastní tréninková data.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativní získávání — Vzor Výrobce-Kontrolora

Klíčovou výhodou Agentic RAG je **iterativní získávání**. Agent může provést několik kol vyhledávání, aby ověřil, zpřesnil nebo rozšířil své počáteční nálezy — podobně jako pracovní postup „výrobce-kontrolor“:

1. **Krok výrobce**: Agent získá počáteční informace a navrhne odpověď.
2. **Krok kontrolora**: Agent provede další získávání, aby ověřil detaily nebo vyplnil mezery.

Níže je agentovi položena otázka, která vyžaduje porovnání několika destinací, což ho vede k opakovanému vyhledávání.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Souhrn

V této lekci jste se naučili, jak postavit systém **Agentic RAG** pomocí Microsoft Agent Framework:

- **Agentic RAG** umožňuje agentům autonomně rozhodovat, kdy získávat informace, čímž se získávání stává dynamickým místo pevného.
- **Nástroje jako zdroje dat**: Externí znalostní báze (jako Azure AI Search) jsou zabalena jako nástroje, které může agent vyvolat.
- **Iterativní získávání**: Vzor maker-checker umožňuje agentovi provádět více kol získávání — vyhledávání, ověřování a zpřesňování — před vytvořením konečné odpovědi.

V produkci byste nahradili paměťovou `TRAVEL_KNOWLEDGE_BASE` skutečným indexem Azure AI Search pro zpracování rozsáhlého vyhledávání cestovních dokumentů.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:  
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho rodném jazyce by měl být považován za závazný zdroj. Pro důležité informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoliv nedorozumění nebo nesprávné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
